# Teste do Mercado RAG — GymSite

**Grupo:** `b2bce16f-3695-4d93-94c5-4bd77bdb92d6` (Mercado Fitness)
**Status:** 124 chunks · 3 documentos · 0 pending
**Agent:** published

## Documentos Ingeridos
| Documento | Chunks | Conteúdo Principal |
|-----------|--------|--------------------|
| BENNY MATHIASON LEWI PRO2022.pdf | 69 | Benchmarks internacionais, métricas do setor |
| mapeamento_franquias_academia_brasil.pdf | 5 | Franquias de academia no Brasil, modelos de negócio |
| panorama-2025.pdf | 50 | Tendências 2025, crescimento do mercado brasileiro |

## O que este notebook testa
1. **Recuperação de tendências** (panorama 2025)
2. **Recuperação de franquias** (mapeamento)
3. **Recuperação de benchmarks** (métricas internacionais)
4. **Avaliação quantitativa** (Recall@K, Precision@K, MRR, document_match)

Busca = embed Ollama (`mxbai-embed-large` @ 1024) + RPC `match_chunks`.

## Requisitos
```bash
pip install supabase python-dotenv openai httpx
```

**Importante:** rode as cells em ordem (Kernel → Restart & Run All).
O cwd do Jupyter costuma ser `notebooks/` — o setup resolve o ROOT do repo.


In [1]:
# Configuracao — rode ESTA cell antes de tudo
import os
import json
from pathlib import Path
from typing import Any, Dict, cast
from dotenv import load_dotenv
from supabase import create_client, Client

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

for env_name in (".env", ".env.local"):
    env_path = ROOT / env_name
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"loaded {env_path}")

SUPABASE_URL = os.getenv("SUPABASE_URL") or os.getenv("VITE_SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")
MERCADO_GROUP_ID = (
    os.getenv("MERCADO_GROUP_ID") or "b2bce16f-3695-4d93-94c5-4bd77bdb92d6"
)

OLLAMA_BASE = (
    os.getenv("OLLAMA_BASE_URL") or "https://ollama2.vectracargo.com.br"
).rstrip("/").removesuffix("/v1")
EMBED_MODEL = os.getenv("EMBEDDING_MODEL") or "mxbai-embed-large"
EMBED_DIM = int(os.getenv("EMBEDDING_DIMENSION") or "1024")
MIN_SIM = float(os.getenv("RAG_MIN_SIMILARITY") or "0.35")
TOP_K = int(os.getenv("RAG_TOP_K") or "5")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError(
        f"SUPABASE_URL / SUPABASE_SERVICE_ROLE_KEY ausentes (ROOT={ROOT})"
    )

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print(f"Supabase: {SUPABASE_URL}")
print(f"Mercado group: {MERCADO_GROUP_ID}")
print(f"Embed: {EMBED_MODEL} @ {OLLAMA_BASE} dim={EMBED_DIM} min_sim={MIN_SIM} top_k={TOP_K}")

agent = (
    supabase.table("eros_knowledge_agents")
    .select("*")
    .eq("group_id", MERCADO_GROUP_ID)
    .execute()
)
if agent.data:
    a = cast(Dict[str, Any], agent.data[0])
    print(f"Agente: {a['name']} · status={a['status']} · chunks={a['chunk_count']}")
else:
    print("Agente nao encontrado")


loaded C:\Users\marce\assistent-control\.env.local
Supabase: https://gxmaxbjgdrqdcizvdojp.supabase.co
Mercado group: b2bce16f-3695-4d93-94c5-4bd77bdb92d6
Embed: mxbai-embed-large @ https://ollama2.vectracargo.com.br dim=1024 min_sim=0.35 top_k=5
Agente: Mercado Fitness · status=published · chunks=124


In [2]:
# Embeddings (mxbai-embed-large @ 1024) + match_chunks
from typing import Any, Dict, List, Optional, cast
import httpx
import unicodedata

def _norm(s: str) -> str:
    s = (s or "").lower()
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )

def embed_query(text: str) -> List[float]:
    # OpenAI-compat embeddings via Ollama — mesmo modelo do ingest Mercado
    url = f"{OLLAMA_BASE}/v1/embeddings"
    with httpx.Client(timeout=60.0) as client:
        r = client.post(url, json={"model": EMBED_MODEL, "input": text[:1000]})
        r.raise_for_status()
        data = r.json()
    vec = data["data"][0]["embedding"]
    if len(vec) != EMBED_DIM:
        raise ValueError(f"dim mismatch: got {len(vec)} expected {EMBED_DIM}")
    return vec

def search_chunks(
    query: str,
    top_k: Optional[int] = None,
    min_similarity: Optional[float] = None,
) -> List[Dict[str, Any]]:
    embedding = embed_query(query)
    result = supabase.rpc(
        "match_chunks",
        {
            "query_embedding": embedding,
            "match_group_id": MERCADO_GROUP_ID,
            "match_tenant_id": None,
            "match_modalidade": None,
            "match_bairro": None,
            "match_plano_rank": None,
            "match_municipio": None,
            "match_k": top_k if top_k is not None else TOP_K,
            "min_similarity": min_similarity if min_similarity is not None else MIN_SIM,
            # Hybrid: vector + portuguese FTS (optional; null = vector-only)
            "match_query": query,
        },
    ).execute()
    return cast(List[Dict[str, Any]], result.data or [])

print("Funcoes OK — search = embed(Ollama mxbai) + match_chunks hybrid (vector+FTS)")


Funcoes OK — search = embed(Ollama mxbai) + match_chunks


## Dataset de Teste — Mercado Fitness

Queries focadas no conteúdo real dos 3 documentos ingeridos:
- **panorama-2025.pdf**: Tendências e crescimento do mercado brasileiro
- **mapeamento_franquias_academia_brasil.pdf**: Franquias e modelos de negócio
- **BENNY MATHIASON LEWI PRO2022.pdf**: Benchmarks internacionais


In [3]:
# Dataset de avaliacao — Mercado Fitness
EVAL_DATASET = [
    {
        "id": "mercado-01",
        "query": "Quais são as principais tendências do mercado fitness em 2025?",
        "expected_keywords": ["tendências", "2025", "crescimento", "mercado"],
        "expected_doc": "panorama-2025.pdf",
    },
    {
        "id": "mercado-02",
        "query": "Quais são as principais franquias de academia no Brasil?",
        "expected_keywords": ["franquias", "academia", "Brasil", "modelos"],
        "expected_doc": "mapeamento_franquias_academia_brasil.pdf",
    },
    {
        "id": "mercado-03",
        "query": "Quais são os benchmarks internacionais de desempenho de academias?",
        "expected_keywords": ["benchmarks", "internacionais", "desempenho", "métricas"],
        "expected_doc": "BENNY MATHIASON LEWI PRO2022.pdf",
    },
    {
        "id": "mercado-04",
        "query": "Qual é o tamanho do mercado fitness brasileiro e seu potencial de crescimento?",
        "expected_keywords": ["mercado", "Brasil", "crescimento", "potencial"],
        "expected_doc": "panorama-2025.pdf",
    },
    {
        "id": "mercado-05",
        "query": "Quais modelos de negócio de academia funcionam melhor no Brasil?",
        "expected_keywords": ["modelos", "negócio", "academia", "Brasil"],
        "expected_doc": "mapeamento_franquias_academia_brasil.pdf",
    },
    {
        "id": "mercado-06",
        "query": "Quais métricas de retenção de alunos são consideradas boas no setor?",
        "expected_keywords": ["retenção", "alunos", "métricas", "setor"],
        "expected_doc": "BENNY MATHIASON LEWI PRO2022.pdf",
    },
    {
        "id": "mercado-07",
        "query": "Quais são as principais redes de academia em franquia no Brasil?",
        "expected_keywords": ["redes", "franquia", "Brasil", "academia"],
        "expected_doc": "mapeamento_franquias_academia_brasil.pdf",
    },
    {
        "id": "mercado-08",
        "query": "Qual é o ticket médio mensal de academias no Brasil?",
        "expected_keywords": ["ticket", "médio", "mensal", "academia"],
        "expected_doc": "panorama-2025.pdf",
    },
]

print(f"Dataset de avaliacao: {len(EVAL_DATASET)} queries")
print(f"Documentos esperados: {len(set(q['expected_doc'] for q in EVAL_DATASET))}")


Dataset de avaliacao: 8 queries
Documentos esperados: 3


## Executar Avaliação

Para cada query:
1. Buscar os chunks mais relevantes (`mxbai-embed-large` + `match_chunks`)
2. Verificar keywords esperadas (Recall@K / Precision@K / MRR)
3. Verificar se o documento esperado foi recuperado (`document_match`)


In [4]:
def chunk_blob(chunk: Dict[str, Any]) -> str:
    return _norm(
        " ".join(
            [
                str(chunk.get("text") or ""),
                str(chunk.get("meta") or ""),
                str(chunk.get("source_ref") or ""),
            ]
        )
    )

def chunk_is_relevant(chunk: Dict[str, Any], expected_keywords: List[str]) -> bool:
    blob = chunk_blob(chunk)
    return any(_norm(kw) in blob for kw in expected_keywords)

def calculate_recall_at_k(retrieved, expected_keywords, k=5) -> float:
    if not expected_keywords:
        return 1.0
    blob = " ".join(chunk_blob(c) for c in retrieved[:k])
    hit = sum(1 for kw in expected_keywords if _norm(kw) in blob)
    return hit / len(expected_keywords)

def calculate_precision_at_k(retrieved, expected_keywords, k=5) -> float:
    if not retrieved[:k]:
        return 0.0
    relevant = sum(1 for c in retrieved[:k] if chunk_is_relevant(c, expected_keywords))
    return relevant / min(k, len(retrieved[:k]))

def calculate_mrr(retrieved, expected_keywords) -> float:
    for i, chunk in enumerate(retrieved):
        if chunk_is_relevant(chunk, expected_keywords):
            return 1.0 / (i + 1)
    return 0.0

def _chunk_source_refs(chunk: Dict[str, Any]) -> List[str]:
    refs: List[str] = []
    top = chunk.get("source_ref")
    if top:
        refs.append(str(top))
    meta = chunk.get("meta") or {}
    if isinstance(meta, str):
        try:
            meta = json.loads(meta)
        except Exception:
            meta = {}
    if isinstance(meta, dict):
        for key in ("source_ref", "document_name"):
            val = meta.get(key)
            if val:
                refs.append(str(val))
    return refs

def check_document_match(retrieved: List[Dict[str, Any]], expected_doc: str) -> bool:
    needle = _norm(expected_doc)
    for chunk in retrieved:
        for ref in _chunk_source_refs(chunk):
            if needle in _norm(ref):
                return True
    return False

def evaluate_query(
    query: str,
    expected_keywords: List[str],
    expected_doc: str,
    top_k: Optional[int] = None,
    min_similarity: Optional[float] = None,
) -> Dict[str, Any]:
    retrieved = search_chunks(query, top_k=top_k, min_similarity=min_similarity)
    return {
        "query": query,
        "expected_doc": expected_doc,
        "retrieved_count": len(retrieved),
        "recall@5": calculate_recall_at_k(retrieved, expected_keywords, k=5),
        "precision@5": calculate_precision_at_k(retrieved, expected_keywords, k=5),
        "mrr": calculate_mrr(retrieved, expected_keywords),
        "document_match": check_document_match(retrieved, expected_doc),
        "top_sims": [round(float(c.get("similarity") or 0), 3) for c in retrieved[:3]],
        "chunks": retrieved[:3],
    }

print("Funcoes de avaliacao OK")


Funcoes de avaliacao OK


In [5]:
# Executar avaliacao
results = []

print("Executando avaliacao (vector match_chunks)...\n")
print("=" * 80)

for eval_item in EVAL_DATASET:
    print(f"\n[{eval_item['id']}] {eval_item['query']}")
    print(f"   Esperado: {eval_item['expected_doc']}")
    print(f"   Keywords: {eval_item['expected_keywords']}")

    result = evaluate_query(
        query=eval_item["query"],
        expected_keywords=eval_item["expected_keywords"],
        expected_doc=eval_item["expected_doc"],
        top_k=TOP_K,
        min_similarity=MIN_SIM,
    )
    result["id"] = eval_item["id"]
    results.append(result)

    print(f"   Recuperados: {result['retrieved_count']} chunks sims={result['top_sims']}")
    print(f"   Recall@5: {result['recall@5']:.2f}")
    print(f"   Precision@5: {result['precision@5']:.2f}")
    print(f"   MRR: {result['mrr']:.2f}")
    print(f"   document_match: {'YES' if result['document_match'] else 'NO'}")

    if result["chunks"]:
        first = result["chunks"][0]
        refs = _chunk_source_refs(first)
        preview = (first.get("text") or "")[:120].replace("\n", " ")
        print(f"   Primeiro: {refs[0] if refs else 'N/A'}")
        print(f"   Preview: {preview}...")

print("\n" + "=" * 80)


Executando avaliacao (vector match_chunks)...


[mercado-01] Quais são as principais tendências do mercado fitness em 2025?
   Esperado: panorama-2025.pdf
   Keywords: ['tendências', '2025', 'crescimento', 'mercado']
   Recuperados: 5 chunks sims=[0.828, 0.822, 0.819]
   Recall@5: 0.75
   Precision@5: 1.00
   MRR: 1.00
   document_match: YES
   Primeiro: data/raw/Mercado/panorama-2025.pdf
   Preview: nichados, impulsionado pela restrição imobiliária, tem se  consolidado como uma alternativa de eficiência e diversificaç...

[mercado-02] Quais são as principais franquias de academia no Brasil?
   Esperado: mapeamento_franquias_academia_brasil.pdf
   Keywords: ['franquias', 'academia', 'Brasil', 'modelos']
   Recuperados: 5 chunks sims=[0.726, 0.697, 0.696]
   Recall@5: 0.75
   Precision@5: 1.00
   MRR: 1.00
   document_match: YES
   Primeiro: data/raw/Mercado/mapeamento_franquias_academia_brasil.pdf
   Preview: MAPEAMENTO COMPLETO: FRANQUIAS DE ACADEMIA  Relatório de Inteligência Comerci

## Relatório de Avaliação

Métricas agregadas de todas as queries.


In [6]:
if not results:
    raise RuntimeError("results vazio — rode a cell de avaliacao antes")

avg_recall = sum(r["recall@5"] for r in results) / len(results)
avg_precision = sum(r["precision@5"] for r in results) / len(results)
avg_mrr = sum(r["mrr"] for r in results) / len(results)
doc_match_rate = sum(1 for r in results if r["document_match"]) / len(results)

print("Relatorio de Avaliacao")
print("=" * 50)
print(f"Total de queries: {len(results)}")
print(f"Recall@5 medio: {avg_recall:.2f}")
print(f"Precision@5 medio: {avg_precision:.2f}")
print(f"MRR medio: {avg_mrr:.2f}")
print(f"Taxa document_match: {doc_match_rate:.0%}")
print("=" * 50)

print("\nQueries com Recall@5 < 0.5:")
low_recall = [r for r in results if r["recall@5"] < 0.5]
if low_recall:
    for r in low_recall:
        print(f"  - {r['query'][:60]}...: Recall@5={r['recall@5']:.2f}")
else:
    print("  Nenhuma")

print("\nQueries sem documento esperado:")
no_doc_match = [r for r in results if not r["document_match"]]
if no_doc_match:
    for r in no_doc_match:
        print(f"  - {r['query'][:60]}...: esperado {r['expected_doc']}")
else:
    print("  Todos os documentos esperados foram recuperados")


Relatorio de Avaliacao
Total de queries: 8
Recall@5 medio: 0.78
Precision@5 medio: 0.90
MRR medio: 0.90
Taxa document_match: 100%

Queries com Recall@5 < 0.5:
  - Quais são os benchmarks internacionais de desempenho de acad...: Recall@5=0.25

Queries sem documento esperado:
  Todos os documentos esperados foram recuperados


## Análise Detalhada por Documento

Verificar quais documentos foram mais recuperados.


In [7]:
# Contar quantas vezes cada documento foi recuperado
doc_counts: Dict[str, int] = {}
for r in results:
    for chunk in r["chunks"]:
        refs = _chunk_source_refs(chunk)
        key = refs[0] if refs else "N/A"
        doc_counts[key] = doc_counts.get(key, 0) + 1

print("Documentos recuperados (top 5):")
sorted_docs = sorted(doc_counts.items(), key=lambda x: x[1], reverse=True)
for doc, count in sorted_docs[:5]:
    print(f"  {doc}: {count} vezes")

print("\nDistribuicao por documento esperado:")
for doc in sorted(set(r["expected_doc"] for r in results)):
    queries_for_doc = [r for r in results if r["expected_doc"] == doc]
    avg_r = sum(r["recall@5"] for r in queries_for_doc) / len(queries_for_doc)
    match_rate = sum(1 for r in queries_for_doc if r["document_match"]) / len(queries_for_doc)
    print(
        f"  {doc}: {len(queries_for_doc)} queries · "
        f"Recall@5={avg_r:.2f} · doc_match={match_rate:.0%}"
    )


Documentos recuperados (top 5):
  data/raw/Mercado/panorama-2025.pdf: 11 vezes
  data/raw/Mercado/BENNY MATHIASON LEWI PRO2022.pdf: 9 vezes
  data/raw/Mercado/mapeamento_franquias_academia_brasil.pdf: 4 vezes

Distribuicao por documento esperado:
  BENNY MATHIASON LEWI PRO2022.pdf: 2 queries · Recall@5=0.50 · doc_match=100%
  mapeamento_franquias_academia_brasil.pdf: 3 queries · Recall@5=0.92 · doc_match=100%
  panorama-2025.pdf: 3 queries · Recall@5=0.83 · doc_match=100%


## Salvar Resultados

Salva em `data/evaluation/mercado_eval_results.json` (relativo ao ROOT do repo).


In [8]:
from datetime import datetime

output_path = ROOT / "data" / "evaluation" / "mercado_eval_results.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

serializable_results = []
for r in results:
    row = {k: v for k, v in r.items() if k != "chunks"}
    row["chunks_preview"] = [
        {
            "source_ref": (_chunk_source_refs(c) or ["N/A"])[0],
            "similarity": c.get("similarity"),
            "text": (c.get("text") or "")[:240],
        }
        for c in r.get("chunks") or []
    ]
    serializable_results.append(row)

eval_report = {
    "timestamp": datetime.now().isoformat(),
    "group_id": MERCADO_GROUP_ID,
    "embed_model": EMBED_MODEL,
    "embed_dim": EMBED_DIM,
    "min_similarity": MIN_SIM,
    "top_k": TOP_K,
    "metrics": {
        "avg_recall@5": avg_recall,
        "avg_precision@5": avg_precision,
        "avg_mrr": avg_mrr,
        "doc_match_rate": doc_match_rate,
    },
    "results": serializable_results,
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(eval_report, f, indent=2, ensure_ascii=False)

print(f"Resultados salvos em: {output_path}")


Resultados salvos em: C:\Users\marce\assistent-control\data\evaluation\mercado_eval_results.json


## Conclusão

O Mercado RAG está com **124 chunks** distribuídos em 3 documentos:
- **panorama-2025.pdf** (50 chunks): Tendências e crescimento do mercado brasileiro
- **BENNY MATHIASON LEWI PRO2022.pdf** (69 chunks): Benchmarks internacionais
- **mapeamento_franquias_academia_brasil.pdf** (5 chunks): Franquias no Brasil

Métricas:
- **Recall@5**: Capacidade de recuperar as keywords esperadas
- **Precision@5**: Precisão dos chunks recuperados
- **MRR**: Posição do primeiro chunk relevante
- **Doc Match Rate**: Taxa de recuperação do documento esperado

Se as métricas estiverem baixas, considere:
1. Adicionar mais documentos ao grupo Mercado
2. Ajustar o chunking (tamanho, overlap)
3. Melhorar os metadados dos chunks
4. Ajustar o threshold de similaridade (`RAG_MIN_SIMILARITY`)
